# 🏡 Fortem Capital — Luxury Demand Model: Los Cabos
### Predicción de Demanda de Compradores de Lujo (EE.UU. & Canadá)

**Objetivo:** Predecir el nivel de demanda de compradores estadounidenses y canadienses de vivienda de lujo en Los Cabos, usando indicadores macroeconómicos públicos + datos reales de turismo (SECTUR), para apoyar la decisión de timing de lanzamiento del nuevo proyecto residencial de Fortem Capital.

**Modelo:** Random Forest Regressor

**Variable Y:** Llegadas reales de turistas EE.UU. + Canadá al aeropuerto de Los Cabos (SECTUR), normalizadas a escala 0–100

**Variables X:** Indicadores macro de EE.UU. obtenidos de FRED + lag features de turismo

**Resultado:** R² = 0.51 — el modelo explica el 51% de la variación en demanda


---
# 📦 Paso 1 — Instalación de librerías


In [1]:
!pip install fredapi --quiet


---
# 📚 Paso 2 — Librerías


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

from fredapi import Fred
from datetime import date
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')


✅ Librerías cargadas correctamente


---
# 📥 Paso 3 — Descarga de datos macroeconómicos (FRED)

FRED es la base de datos pública de la Reserva Federal de EE.UU. con miles de series históricas gratuitas.

| Variable | Serie FRED | Lógica de negocio |
|---|---|---|
| S&P 500 | `SP500` | Efecto riqueza: portafolio alto → comprador americano compra segunda vivienda |
| Confianza del consumidor USA | `UMCSENT` | Si confía en la economía, gasta en lujo |
| Tasa hipotecaria 30 años | `MORTGAGE30US` | Si sube, baja el apetito comprador en EE.UU. |
| Tipo de cambio USD/MXN | `DEXMXUS` | Peso débil = Los Cabos más barato para el extranjero |


In [3]:
FRED_API_KEY = '3974d4fa0e06d76c06d53bf5dea70110'   # Regístrate gratis en: https://fred.stlouisfed.org/docs/api/api_key.html
fred = Fred(api_key=FRED_API_KEY)

START = '2015-01-01'
END   = date.today().strftime('%Y-%m-%d')   # Siempre usa datos hasta hoy

print('📡 Descargando datos de FRED...')

sp500       = fred.get_series('SP500',        observation_start=START, observation_end=END).resample('MS').mean()
confianza   = fred.get_series('UMCSENT',      observation_start=START, observation_end=END).resample('MS').mean()
hipotecas   = fred.get_series('MORTGAGE30US', observation_start=START, observation_end=END).resample('MS').mean()
tipo_cambio = fred.get_series('DEXMXUS',      observation_start=START, observation_end=END).resample('MS').mean()

print('✅ FRED descargado correctamente')
print(f'   S&P 500:          {len(sp500)} meses')
print(f'   Confianza USA:    {len(confianza)} meses')
print(f'   Tasa hipotecaria: {len(hipotecas)} meses')
print(f'   Tipo de cambio:   {len(tipo_cambio)} meses')


📡 Descargando datos de FRED...
✅ FRED descargado correctamente
   S&P 500:          121 meses
   Confianza USA:    134 meses
   Tasa hipotecaria: 135 meses
   Tipo de cambio:   135 meses


---
# 📤 Paso 4 — Cargar datos de SECTUR (Variable Y)

**¿Por qué SECTUR y no Google Trends?**

Google Trends resultó tener demasiados ceros para términos de búsqueda nicho como 'Los Cabos real estate' — no tenía suficiente volumen mensual consistente para ser una Y válida.

SECTUR provee **llegadas reales de turistas** al aeropuerto de Los Cabos, filtradas por residencia EE.UU. y Canadá — exactamente el mercado objetivo de Fortem Capital.

Fuente: [datatur.sectur.gob.mx](https://datatur.sectur.gob.mx) → Llegadas de turistas extranjeros por residencia → Los Cabos → EE.UU. + Canadá


In [5]:
import pandas as pd

# Ruta al archivo (ajústala según dónde esté tu archivo)
file_path = "/Users/paobecerril/Desktop/Fortem_app/BD_Residencia_Limpia.xlsx"

# Leer el archivo
df = pd.read_excel(file_path)

In [ ]:
# Leer el archivo
nombre_archivo = list(uploaded.keys())[0]
sectur = pd.read_excel(nombre_archivo)

print(f'📊 Shape: {sectur.shape}')
print(f'\n📋 Columnas: {sectur.columns.tolist()}')
sectur.head()


In [ ]:
# Agrupamos por mes sumando Hombres + Mujeres de EE.UU. y Canadá
turistas = (
    sectur
    .groupby('Fecha')['Valor']
    .sum()
    .reset_index()
)

turistas['Fecha'] = pd.to_datetime(turistas['Fecha'])
turistas = turistas.set_index('Fecha').sort_index()
turistas.columns = ['turistas_totales']

print(f'✅ Serie de turistas lista: {len(turistas)} meses')
print(f'📅 Periodo: {turistas.index.min().date()} → {turistas.index.max().date()}')
print(f'\n📊 Distribución:')
print(turistas.describe())


---
# 🧱 Paso 5 — Construcción del DataFrame

Unimos los datos macro de FRED con los datos de turismo de SECTUR en un solo DataFrame, una fila por mes.


In [ ]:
df = pd.DataFrame({
    'sp500'           : sp500,
    'confianza'       : confianza,
    'hipotecas'       : hipotecas,
    'tipo_cambio'     : tipo_cambio,
    'turistas_totales': turistas['turistas_totales']
})

df = df.dropna()
df['mes']       = df.index.month
df['trimestre'] = df.index.quarter

print(f'📊 Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📅 Periodo: {df.index.min().date()} → {df.index.max().date()}')
df.head(10)


---
# 🎯 Paso 6 — Construcción de Y + Exclusión de COVID + Lag Features

**3 decisiones clave en este paso:**

1. **Excluir COVID (Mar 2020 – Dic 2021):** El colapso de turismo fue una anomalía exógena, no un patrón de mercado. Incluirlo distorsiona el aprendizaje del modelo.

2. **Normalizar Y a escala 0–100:** Para que el índice sea interpretable independientemente del volumen absoluto.

3. **Lag features:** Le damos memoria al modelo — cuántos turistas vinieron el mes pasado, hace 3 meses, y el mismo mes del año anterior. Esto mejoró el R² de 0.01 a 0.51.


In [ ]:
# ── Excluir periodo COVID ────────────────────────────────────────────────────
df_clean = df[~((df.index >= '2020-03-01') & (df.index <= '2021-12-01'))].copy()

print(f'Filas originales:  {len(df)}')
print(f'Filas sin COVID:   {len(df_clean)}')
print(f'Meses excluidos:   {len(df) - len(df_clean)} (Mar 2020 – Dic 2021)')

# ── Normalizar Y ─────────────────────────────────────────────────────────────
def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min()) * 100

df_clean['demand_index'] = normalizar(df_clean['turistas_totales'])

# ── Lag features (memoria temporal) ──────────────────────────────────────────
df_clean['turistas_lag1']  = df_clean['turistas_totales'].shift(1)   # mes anterior
df_clean['turistas_lag3']  = df_clean['turistas_totales'].shift(3)   # hace 3 meses
df_clean['turistas_lag12'] = df_clean['turistas_totales'].shift(12)  # mismo mes año anterior

df_clean = df_clean.dropna()

# Renormalizar Y después de dropna
df_clean['demand_index'] = normalizar(df_clean['turistas_totales'])

print(f'\n✅ Dataset final listo: {len(df_clean)} filas')
print(f'\n📊 Distribución de Y (demand_index):')
print(df_clean['demand_index'].describe())


---
# 🔍 Paso 7 — Análisis Exploratorio (EDA)


In [ ]:
# ── Índice de demanda en el tiempo ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_clean.index, df_clean['demand_index'], color='#1a3c5e', linewidth=2)
ax.fill_between(df_clean.index, df_clean['demand_index'], alpha=0.15, color='#1a3c5e')
ax.axhline(df_clean['demand_index'].mean(), color='orange', linestyle='--', label='Promedio histórico')
ax.set_title('📈 Índice de Demanda Luxury — Los Cabos (sin COVID)', fontsize=14, fontweight='bold')
ax.set_ylabel('Índice (0–100)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Estacionalidad por mes ────────────────────────────────────────────────────
demanda_mes   = df_clean.groupby('mes')['demand_index'].mean()
meses_nombres = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(meses_nombres, demanda_mes.values, color='#1a3c5e', alpha=0.85)
max_idx = demanda_mes.values.argmax()
bars[max_idx].set_color('#c9a84c')

for bar, val in zip(bars, demanda_mes.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('📅 Demanda Promedio por Mes — Estacionalidad', fontsize=13, fontweight='bold')
ax.set_ylabel('Índice Promedio (0–100)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\n🏆 Mes de mayor demanda histórica: {meses_nombres[max_idx]}')


In [ ]:
# ── Correlación entre variables ───────────────────────────────────────────────
feature_cols_corr = ['sp500', 'confianza', 'hipotecas', 'tipo_cambio', 'mes', 'trimestre', 'demand_index']
corr = df_clean[feature_cols_corr].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title('🔗 Correlación entre Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== CORRELACIÓN X vs Y ===')
print(corr['demand_index'].drop('demand_index').sort_values(ascending=False))


---
# ⚙️ Paso 8 — Definir X e Y


In [ ]:
# Variables predictoras — macro + memoria temporal
feature_cols = [
    'sp500', 'confianza', 'hipotecas', 'tipo_cambio',  # Variables macro de EE.UU.
    'mes', 'trimestre',                                  # Estacionalidad
    'turistas_lag1', 'turistas_lag3', 'turistas_lag12'  # Memoria temporal (lag features)
]

X = df_clean[feature_cols]
y = df_clean['demand_index']

print(f'✅ X shape: {X.shape}  →  {X.shape[0]} meses × {X.shape[1]} variables')
print(f'✅ Y shape: {y.shape}')
print(f'\n📋 Variables X:')
for col in feature_cols:
    print(f'   • {col}')
print(f'\n🎯 Variable Y: demand_index (turistas EE.UU.+Canadá, normalizado 0–100)')


---
# ✂️ Paso 9 — Train-Test Split

**shuffle=False** porque son series de tiempo — no mezclamos el pasado con el futuro.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=False   # Crítico para series de tiempo
)

print(f'📚 Training set: {X_train.shape[0]} meses  ({X_train.index.min().date()} → {X_train.index.max().date()})')
print(f'🧪 Test set:     {X_test.shape[0]} meses   ({X_test.index.min().date()} → {X_test.index.max().date()})')


---
# 🤖 Paso 10 — Pipeline + Random Forest Regressor

Mismo patrón de Pipeline que el notebook de Bike Sharing.


In [ ]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', feature_cols)  # Random Forest no necesita scaling
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,   # Encontrado como óptimo tras evaluación
        max_depth=5,        # Limita profundidad para evitar sobreajuste
        random_state=42,
        n_jobs=-1
    ))
])

print('✅ Pipeline creado')
print('   • n_estimators = 200')
print('   • max_depth    = 5')


---
# 🏋️ Paso 11 — Entrenamiento


In [ ]:
print('🏋️ Entrenando el modelo...')
rf_pipeline.fit(X_train, y_train)
print('✅ Modelo entrenado exitosamente')


---
# 📊 Paso 12 — Evaluación del Modelo


In [ ]:
y_pred = rf_pipeline.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = np.mean(np.abs(y_test - y_pred))

print('='*50)
print('📊 RESULTADOS DEL MODELO')
print('='*50)
print(f'   R²   = {r2:.4f}  →  explica el {r2*100:.1f}% de la variación en demanda')
print(f'   RMSE = {rmse:.2f}  →  error promedio de {rmse:.1f} puntos en el índice')
print(f'   MAE  = {mae:.2f}')
print('='*50)

if r2 > 0.4:
    print('\n✅ Modelo APROBADO — Capacidad predictiva aceptable para 84 observaciones')
else:
    print('\n⚠️ Modelo mejorable')


In [ ]:
# ── Real vs Predicho ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(y_test.index, y_test.values,  label='Demanda Real',     color='#1a3c5e', linewidth=2)
ax.plot(y_test.index, y_pred,          label='Demanda Predicha', color='#c9a84c', linewidth=2, linestyle='--')
ax.set_title('🎯 Demanda Real vs Predicha (Test Set)', fontsize=13, fontweight='bold')
ax.set_ylabel('Índice de Demanda (0–100)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
# 🌟 Paso 13 — Feature Importance

¿Qué variables explican más la demanda de turistas de lujo en Los Cabos?


In [ ]:
importancias = rf_pipeline.named_steps['model'].feature_importances_

nombres_display = {
    'sp500'           : 'S&P 500 (EE.UU.)',
    'confianza'       : 'Confianza del Consumidor USA',
    'hipotecas'       : 'Tasa Hipotecaria 30 años',
    'tipo_cambio'     : 'Tipo de Cambio USD/MXN',
    'mes'             : 'Mes del año (estacionalidad)',
    'trimestre'       : 'Trimestre',
    'turistas_lag1'   : 'Turistas mes anterior (lag 1)',
    'turistas_lag3'   : 'Turistas hace 3 meses (lag 3)',
    'turistas_lag12'  : 'Turistas mismo mes año anterior (lag 12)'
}

fi_df = pd.DataFrame({
    'Variable'   : [nombres_display[c] for c in feature_cols],
    'Importancia': importancias
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#c9a84c' if v == fi_df['Importancia'].max() else '#1a3c5e' for v in fi_df['Importancia']]
ax.barh(fi_df['Variable'], fi_df['Importancia'], color=colores)

for i, val in enumerate(fi_df['Importancia']):
    ax.text(val + 0.002, i, f'{val*100:.1f}%', va='center', fontsize=9)

ax.set_title('🌟 ¿Qué variables explican más la demanda de lujo en Los Cabos?',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia relativa')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

top_var = fi_df.iloc[-1]['Variable']
print(f'\n🏆 Variable más influyente: {top_var}')
print('\n💼 Interpretación de negocio:')
print('   lag12  → El mismo mes del año anterior es el mejor predictor: hay fuerte estacionalidad anual')
print('   lag1   → La demanda del mes anterior influye en la demanda actual')
print('   S&P    → Efecto riqueza: mercado accionario alto → más viajes de lujo')
print('   USD/MXN→ Peso débil = Los Cabos más accesible para el americano')


---
# 🔮 Paso 14 — Forecast: ¿Cuál es el mejor momento para lanzar?


In [ ]:
# Valores actuales como base del forecast
ultimo_sp500       = df_clean['sp500'].iloc[-1]
ultimo_confianza   = df_clean['confianza'].iloc[-1]
ultimo_hipotecas   = df_clean['hipotecas'].iloc[-1]
ultimo_tipo_cambio = df_clean['tipo_cambio'].iloc[-1]
ultimo_lag1        = df_clean['turistas_totales'].iloc[-1]
ultimo_lag3        = df_clean['turistas_totales'].iloc[-3]
ultimo_lag12       = df_clean['turistas_totales'].iloc[-12]

print(f'📌 Condiciones actuales (último mes disponible):')
print(f'   S&P 500:          {ultimo_sp500:,.0f}')
print(f'   Confianza USA:    {ultimo_confianza:.1f}')
print(f'   Tasa hipotecaria: {ultimo_hipotecas:.2f}%')
print(f'   Tipo de cambio:   {ultimo_tipo_cambio:.2f} MXN/USD')

# Crear próximos 12 meses
fechas_futuras = pd.date_range(
    start=df_clean.index[-1] + pd.DateOffset(months=1),
    periods=12,
    freq='MS'
)

forecast_df = pd.DataFrame({
    'sp500'           : ultimo_sp500,
    'confianza'       : ultimo_confianza,
    'hipotecas'       : ultimo_hipotecas,
    'tipo_cambio'     : ultimo_tipo_cambio,
    'mes'             : fechas_futuras.month,
    'trimestre'       : fechas_futuras.quarter,
    'turistas_lag1'   : ultimo_lag1,
    'turistas_lag3'   : ultimo_lag3,
    'turistas_lag12'  : ultimo_lag12
}, index=fechas_futuras)

forecast_pred = rf_pipeline.predict(forecast_df[feature_cols])
forecast_df['demand_index_pred'] = forecast_pred

print(f'\n✅ Forecast completado para {len(fechas_futuras)} meses')


In [ ]:
meses_es = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
            7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

fig, ax = plt.subplots(figsize=(13, 5))

hist = df_clean['demand_index'].tail(24)
ax.plot(hist.index, hist.values, color='#1a3c5e', linewidth=2, label='Histórico')
ax.plot(forecast_df.index, forecast_df['demand_index_pred'],
        color='#c9a84c', linewidth=2.5, linestyle='--', label='Forecast 12 meses')
ax.fill_between(forecast_df.index, forecast_df['demand_index_pred'], alpha=0.15, color='#c9a84c')

mejor_mes_idx = forecast_df['demand_index_pred'].idxmax()
ax.axvline(mejor_mes_idx, color='green', linestyle=':', alpha=0.8, linewidth=2, label='Mejor mes para lanzar')
ax.axhline(60, color='green',  linestyle=':', alpha=0.4)
ax.axhline(40, color='orange', linestyle=':', alpha=0.4)

ax.set_title('🔮 Forecast de Demanda Luxury — Los Cabos (próximos 12 meses)', fontsize=13, fontweight='bold')
ax.set_ylabel('Índice de Demanda (0–100)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejor_demanda  = forecast_df['demand_index_pred'].max()
mejor_mes_num  = mejor_mes_idx.month

print(f'\n🏆 RECOMENDACIÓN DE TIMING PARA FORTEM CAPITAL:')
print(f'   Mejor mes para lanzar: {meses_es[mejor_mes_num]} {mejor_mes_idx.year}')
print(f'   Índice de demanda proyectado: {mejor_demanda:.1f}/100')

if mejor_demanda >= 60:
    print('   Semáforo: 🟢 ALTA DEMANDA — Momento óptimo para lanzar')
elif mejor_demanda >= 40:
    print('   Semáforo: 🟡 DEMANDA MEDIA — Evaluar con cautela')
else:
    print('   Semáforo: 🔴 DEMANDA BAJA — Esperar mejores condiciones macro')


---
# 💾 Paso 15 — Guardar el modelo


In [ ]:
import joblib

joblib.dump(rf_pipeline, 'fortem_rf_pipeline.pkl')
df_clean.to_csv('fortem_demand_data.csv')
forecast_df.to_csv('fortem_forecast.csv')

print('✅ fortem_rf_pipeline.pkl')
print('✅ fortem_demand_data.csv')
print('✅ fortem_forecast.csv')


---
# 🚀 Paso 16 — Dashboard Streamlit (Versión Profesional Fortem Capital)


In [ ]:
%%writefile fortem_app.py
"""
Fortem Capital — Luxury Demand Intelligence
Los Cabos Residential Investment Analysis

Desarrollado por: Ana Paola Becerril Gutiérrez
"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title='Fortem Capital · Los Cabos Intelligence',
    page_icon='🏛️',
    layout='wide',
    initial_sidebar_state='expanded'
)

# ─────────────────────────────────────────────
# BRAND PALETTE
# ─────────────────────────────────────────────
NAVY   = '#0B1F3A'   # Fortem navy (primary)
GOLD   = '#C9A84C'   # Luxury gold (accent)
STEEL  = '#2C4A6E'   # Mid blue
CREAM  = '#F5F0E8'   # Warm off-white
SLATE  = '#64748B'   # Muted text
GREEN  = '#2D6A4F'   # Semáforo verde
ORANGE = '#CA6702'   # Semáforo amarillo
RED    = '#9B2226'   # Semáforo rojo

# ─────────────────────────────────────────────
# GLOBAL CSS — Corporate & Refined
# ─────────────────────────────────────────────
st.markdown(f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:wght@300;400;500;600;700&family=DM+Sans:wght@300;400;500&display=swap');

  /* ── Global ── */
  html, body, [class*="css"] {{
    font-family: 'DM Sans', sans-serif;
    background-color: #FAFAF8;
    color: {NAVY};
  }}

  /* ── Hide default streamlit chrome ── */
  #MainMenu, footer, header {{ visibility: hidden; }}

  /* ── Sidebar ── */
  [data-testid="stSidebar"] {{
    background: {NAVY};
    border-right: 1px solid {GOLD}33;
  }}
  [data-testid="stSidebar"] * {{
    color: {CREAM} !important;
  }}
  [data-testid="stSidebar"] .stMarkdown h1,
  [data-testid="stSidebar"] .stMarkdown h2,
  [data-testid="stSidebar"] .stMarkdown h3 {{
    color: {GOLD} !important;
    font-family: 'Cormorant Garamond', serif;
  }}
  [data-testid="stSidebar"] hr {{
    border-color: {GOLD}44;
  }}
  [data-testid="stSidebar"] .stSlider > div > div > div > div {{
    background: {GOLD};
  }}
  [data-testid="stSidebar"] label {{
    color: {CREAM} !important;
    font-size: 0.82rem !important;
    letter-spacing: 0.03em;
  }}
  [data-testid="stSidebar"] .stSelectbox > div > div {{
    background: {STEEL};
    color: {CREAM};
    border: 1px solid {GOLD}55;
  }}

  /* ── Hero header ── */
  .hero-header {{
    background: linear-gradient(135deg, {NAVY} 0%, {STEEL} 100%);
    padding: 2.5rem 3rem;
    border-radius: 0 0 16px 16px;
    margin: -1rem -1rem 2rem -1rem;
    position: relative;
    overflow: hidden;
  }}
  .hero-header::before {{
    content: '';
    position: absolute;
    top: -60px; right: -60px;
    width: 220px; height: 220px;
    border-radius: 50%;
    background: {GOLD}18;
    pointer-events: none;
  }}
  .hero-header::after {{
    content: '';
    position: absolute;
    bottom: -40px; left: 200px;
    width: 120px; height: 120px;
    border-radius: 50%;
    background: {GOLD}10;
    pointer-events: none;
  }}
  .hero-title {{
    font-family: 'Cormorant Garamond', serif;
    font-size: 2.6rem;
    font-weight: 600;
    color: #FFFFFF;
    letter-spacing: 0.01em;
    margin: 0 0 0.25rem 0;
    line-height: 1.1;
  }}
  .hero-subtitle {{
    font-family: 'DM Sans', sans-serif;
    font-size: 1rem;
    color: {GOLD};
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin: 0 0 0.6rem 0;
    font-weight: 400;
  }}
  .hero-tagline {{
    font-family: 'DM Sans', sans-serif;
    font-size: 0.85rem;
    color: rgba(255,255,255,0.65);
    margin: 0;
    font-weight: 300;
    letter-spacing: 0.02em;
  }}
  .hero-badge {{
    display: inline-block;
    background: {GOLD}22;
    border: 1px solid {GOLD}66;
    color: {GOLD};
    font-size: 0.72rem;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    padding: 0.25rem 0.8rem;
    border-radius: 20px;
    margin-top: 0.8rem;
    font-family: 'DM Sans', sans-serif;
  }}

  /* ── Section titles ── */
  .section-title {{
    font-family: 'Cormorant Garamond', serif;
    font-size: 1.6rem;
    font-weight: 600;
    color: {NAVY};
    margin: 2rem 0 0.15rem 0;
    letter-spacing: 0.01em;
  }}
  .section-subtitle {{
    font-family: 'DM Sans', sans-serif;
    font-size: 0.88rem;
    color: {SLATE};
    margin: 0 0 1.2rem 0;
    font-weight: 300;
  }}

  /* ── KPI cards ── */
  .kpi-card {{
    background: white;
    border: 1px solid #E8E3DB;
    border-top: 3px solid {GOLD};
    border-radius: 10px;
    padding: 1.2rem 1.4rem;
    box-shadow: 0 2px 12px rgba(11,31,58,0.07);
  }}
  .kpi-label {{
    font-size: 0.72rem;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: {SLATE};
    font-weight: 500;
    margin-bottom: 0.35rem;
  }}
  .kpi-value {{
    font-family: 'Cormorant Garamond', serif;
    font-size: 2rem;
    font-weight: 600;
    color: {NAVY};
    line-height: 1;
  }}
  .kpi-delta {{
    font-size: 0.78rem;
    color: {SLATE};
    margin-top: 0.3rem;
  }}
  .kpi-delta.positive {{ color: {GREEN}; }}
  .kpi-delta.negative {{ color: {RED}; }}

  /* ── Semáforo badge ── */
  .semaforo-green {{
    background: #D1FAE5; color: {GREEN};
    border: 1px solid #6EE7B7;
    border-radius: 8px; padding: 0.6rem 1rem;
    font-weight: 600; font-size: 0.9rem;
  }}
  .semaforo-yellow {{
    background: #FEF3C7; color: {ORANGE};
    border: 1px solid #FCD34D;
    border-radius: 8px; padding: 0.6rem 1rem;
    font-weight: 600; font-size: 0.9rem;
  }}
  .semaforo-red {{
    background: #FEE2E2; color: {RED};
    border: 1px solid #FCA5A5;
    border-radius: 8px; padding: 0.6rem 1rem;
    font-weight: 600; font-size: 0.9rem;
  }}

  /* ── Insight box ── */
  .insight-box {{
    background: linear-gradient(135deg, {NAVY}08, {GOLD}0A);
    border-left: 3px solid {GOLD};
    border-radius: 0 8px 8px 0;
    padding: 1rem 1.2rem;
    margin: 1rem 0;
    font-size: 0.88rem;
    color: {NAVY};
    font-weight: 400;
    line-height: 1.6;
  }}
  .insight-box strong {{ color: {NAVY}; font-weight: 600; }}

  /* ── Source pill ── */
  .source-pill {{
    display: inline-block;
    background: {NAVY}0D;
    border: 1px solid {NAVY}22;
    border-radius: 20px;
    padding: 0.2rem 0.65rem;
    font-size: 0.72rem;
    color: {NAVY};
    letter-spacing: 0.06em;
    text-transform: uppercase;
    margin: 0 0.25rem 0.4rem 0;
    font-weight: 500;
  }}

  /* ── Info tooltip card ── */
  .tooltip-card {{
    background: white;
    border: 1px solid #E8E3DB;
    border-radius: 10px;
    padding: 1rem 1.2rem;
    font-size: 0.83rem;
    color: {SLATE};
    line-height: 1.6;
    box-shadow: 0 2px 8px rgba(11,31,58,0.06);
  }}
  .tooltip-card .label {{
    font-size: 0.7rem;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: {GOLD};
    font-weight: 600;
    margin-bottom: 0.4rem;
  }}

  /* ── Metrics override ── */
  [data-testid="metric-container"] {{
    background: white;
    border: 1px solid #E8E3DB;
    border-top: 3px solid {GOLD};
    border-radius: 10px;
    padding: 0.9rem 1rem;
    box-shadow: 0 2px 8px rgba(11,31,58,0.06);
  }}
  [data-testid="metric-container"] label {{
    font-size: 0.72rem !important;
    letter-spacing: 0.07em !important;
    text-transform: uppercase !important;
    color: {SLATE} !important;
  }}
  [data-testid="metric-container"] [data-testid="stMetricValue"] {{
    font-family: 'Cormorant Garamond', serif !important;
    font-size: 1.9rem !important;
    color: {NAVY} !important;
  }}

  /* ── Footer ── */
  .footer {{
    margin-top: 4rem;
    padding: 1.5rem 0 1rem 0;
    border-top: 1px solid #E8E3DB;
    text-align: center;
    color: {SLATE};
    font-size: 0.78rem;
    letter-spacing: 0.02em;
    line-height: 1.8;
  }}
  .footer .name {{
    font-family: 'Cormorant Garamond', serif;
    font-size: 0.95rem;
    color: {NAVY};
    font-weight: 600;
    letter-spacing: 0.04em;
  }}
  .footer .divider {{
    color: {GOLD};
    margin: 0 0.5rem;
  }}

  /* ── Expander ── */
  .streamlit-expanderHeader {{
    font-size: 0.83rem !important;
    color: {NAVY} !important;
    font-weight: 500 !important;
    letter-spacing: 0.02em;
  }}
  .streamlit-expanderContent {{
    font-size: 0.83rem;
    color: {SLATE};
    line-height: 1.7;
  }}

  /* ── Plot containers ── */
  [data-testid="stImage"] {{
    border-radius: 10px;
    overflow: hidden;
  }}

  /* ── Divider ── */
  .gold-divider {{
    border: none;
    height: 1px;
    background: linear-gradient(90deg, transparent, {GOLD}88, transparent);
    margin: 2rem 0;
  }}
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
MESES_ES = {
    1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril',
    5:'Mayo', 6:'Junio', 7:'Julio', 8:'Agosto',
    9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'
}
MESES_CORTO = {
    1:'Ene', 2:'Feb', 3:'Mar', 4:'Abr', 5:'May', 6:'Jun',
    7:'Jul', 8:'Ago', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dic'
}

FEATURE_COLS = [
    'sp500', 'confianza', 'hipotecas', 'tipo_cambio',
    'mes', 'trimestre', 'turistas_lag1', 'turistas_lag3', 'turistas_lag12'
]

def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min()) * 100

def semaforo_html(valor):
    if valor >= 60:
        return f'<div class="semaforo-green">🟢 ALTA DEMANDA — Condiciones óptimas para lanzar</div>'
    elif valor >= 40:
        return f'<div class="semaforo-yellow">🟡 DEMANDA MEDIA — Evaluar con cautela antes de decidir</div>'
    else:
        return f'<div class="semaforo-red">🔴 DEMANDA BAJA — Esperar mejores condiciones de mercado</div>'

def set_plot_style(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor('#FAFAF8')
    ax.figure.set_facecolor('#FAFAF8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E8E3DB')
    ax.spines['bottom'].set_color('#E8E3DB')
    ax.tick_params(colors=SLATE, labelsize=8.5)
    ax.yaxis.label.set_color(SLATE)
    ax.xaxis.label.set_color(SLATE)
    ax.set_ylabel(ylabel, fontsize=9, color=SLATE, labelpad=8)
    ax.set_xlabel(xlabel, fontsize=9, color=SLATE, labelpad=8)
    if title:
        ax.set_title(title, fontsize=11, fontweight='600', color=NAVY, pad=12,
                     fontfamily='DejaVu Sans')
    ax.grid(True, axis='y', alpha=0.4, color='#E8E3DB', linewidth=0.8)
    ax.grid(False, axis='x')

# ─────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────
@st.cache_resource(show_spinner='Cargando modelo de inteligencia...')
def cargar_modelo():
    return joblib.load('fortem_rf_pipeline.pkl')

@st.cache_data(show_spinner=False)
def cargar_datos():
    df       = pd.read_csv('fortem_demand_data.csv',  index_col=0, parse_dates=True)
    forecast = pd.read_csv('fortem_forecast.csv',     index_col=0, parse_dates=True)
    return df, forecast

try:
    rf_pipeline     = cargar_modelo()
    df, forecast_df = cargar_datos()
    DATA_LOADED = True
except Exception as e:
    DATA_LOADED = False
    st.error(f'⚠️ No se encontraron los archivos del modelo. Ejecuta primero el notebook para generarlos. ({e})')
    st.stop()

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.markdown('<div style="text-align:center;padding:1rem 0 0.5rem 0;">', unsafe_allow_html=True)
    st.markdown('<h2 style="font-family:Cormorant Garamond,serif;font-size:1.4rem;letter-spacing:0.04em;margin:0;">FORTEM CAPITAL</h2>', unsafe_allow_html=True)
    st.markdown('<p style="font-size:0.7rem;letter-spacing:0.15em;color:#C9A84C;margin:0.2rem 0 0.8rem 0;text-transform:uppercase;">Los Cabos · Análisis de Demanda</p>', unsafe_allow_html=True)
    st.markdown('</div>', unsafe_allow_html=True)

    st.markdown('---')
    st.markdown('### 🎛️ Simulador de Escenarios')
    st.markdown('<p style="font-size:0.78rem;opacity:0.7;margin-bottom:1rem;">Ajusta las condiciones del mercado para explorar distintos escenarios de demanda.</p>', unsafe_allow_html=True)

    sp500_sim = st.slider(
        '📈 S&P 500', 3000, 7000,
        int(df['sp500'].iloc[-1]), step=100,
        help='Índice bursátil de EE.UU. Un S&P alto refleja riqueza del comprador objetivo.'
    )
    confianza_sim = st.slider(
        '😊 Confianza del Consumidor USA', 50, 110,
        int(df['confianza'].iloc[-1]),
        help='Índice de confianza económica de los hogares americanos (U. of Michigan). Mayor confianza = más gasto en lujo.'
    )
    hipotecas_sim = st.slider(
        '🏦 Tasa Hipotecaria 30 años (%)', 3.0, 9.0,
        float(df['hipotecas'].iloc[-1]), step=0.1,
        help='Las tasas altas reducen el apetito comprador. Aunque Los Cabos es cash-heavy, el sentimiento hipotecario influye.'
    )
    tipo_cambio_sim = st.slider(
        '💱 Tipo de Cambio USD/MXN', 15.0, 22.0,
        float(df['tipo_cambio'].iloc[-1]), step=0.1,
        help='Un peso más débil hace que Los Cabos sea más atractivo en precio relativo para el comprador americano.'
    )
    mes_sim = st.selectbox(
        '📅 Mes de Lanzamiento',
        options=list(range(1, 13)),
        format_func=lambda m: MESES_ES[m],
        index=0
    )
    lag1_sim = st.slider(
        '👥 Turistas recientes (mes anterior)', 50000, 300000,
        int(df['turistas_totales'].iloc[-1]), step=5000,
        help='Volumen de turistas EE.UU. + Canadá en el último mes. Es el mejor proxy de momentum actual de la demanda.'
    )

    st.markdown('---')
    st.markdown('<p style="font-size:0.7rem;opacity:0.55;text-align:center;line-height:1.6;">Fuentes · FRED Federal Reserve · SECTUR México<br>Modelo · Random Forest Regressor<br>R² = 0.51</p>', unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SIDEBAR PREDICTION
# ─────────────────────────────────────────────
input_sim = pd.DataFrame([{
    'sp500'           : sp500_sim,
    'confianza'       : confianza_sim,
    'hipotecas'       : hipotecas_sim,
    'tipo_cambio'     : tipo_cambio_sim,
    'mes'             : mes_sim,
    'trimestre'       : (mes_sim - 1) // 3 + 1,
    'turistas_lag1'   : lag1_sim,
    'turistas_lag3'   : lag1_sim,
    'turistas_lag12'  : lag1_sim
}])
pred_sim = rf_pipeline.predict(input_sim[FEATURE_COLS])[0]

# Pre-compute KPIs
demanda_actual    = df['demand_index'].iloc[-1]
demanda_historica = df['demand_index'].mean()
demanda_forecast  = forecast_df['demand_index_pred'].max()
mejor_mes         = forecast_df['demand_index_pred'].idxmax()
delta_actual      = demanda_actual - demanda_historica
delta_sign        = '+' if delta_actual >= 0 else ''

# ─────────────────────────────────────────────
# HERO HEADER
# ─────────────────────────────────────────────
st.markdown(f"""
<div class="hero-header">
  <p class="hero-subtitle">Fortem Capital · Investment Intelligence</p>
  <h1 class="hero-title">Los Cabos Luxury<br>Demand Model</h1>
  <p class="hero-tagline">
    Análisis cuantitativo de demanda de compradores de segunda vivienda de lujo · EE.UU. &amp; Canadá
  </p>
  <span class="hero-badge">🏛️ Confidencial · Uso Interno</span>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# KPIs ROW
# ─────────────────────────────────────────────
st.markdown('<p class="section-title">Indicadores Clave</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">Pulso actual del mercado de demanda de lujo en Los Cabos.</p>', unsafe_allow_html=True)

k1, k2, k3, k4 = st.columns(4)
with k1:
    st.metric(
        label='📍 Demanda Actual',
        value=f'{demanda_actual:.1f} / 100',
        delta=f'{delta_sign}{delta_actual:.1f} pts vs promedio histórico'
    )
with k2:
    st.metric(
        label='🔮 Pico Proyectado (12 meses)',
        value=f'{demanda_forecast:.1f} / 100'
    )
with k3:
    st.metric(
        label='📅 Mejor Mes para Lanzar',
        value=f'{MESES_CORTO[mejor_mes.month]} {mejor_mes.year}'
    )
with k4:
    semaforo_label = '🟢 LANZAR' if demanda_forecast >= 60 else '🟡 EVALUAR' if demanda_forecast >= 40 else '🔴 ESPERAR'
    st.metric(label='🚦 Recomendación', value=semaforo_label)

# ─────────────────────────────────────────────
# SECTION 1 — SIMULADOR RESULTADO
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">🎛️ Resultado del Escenario Simulado</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">Basado en los parámetros que seleccionaste en el panel izquierdo.</p>', unsafe_allow_html=True)

r1, r2, r3 = st.columns([1.2, 1.8, 1])
with r1:
    color_hex = GREEN if pred_sim >= 60 else ORANGE if pred_sim >= 40 else RED
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 1rem;background:white;
                border-radius:12px;border:1px solid #E8E3DB;
                box-shadow:0 2px 12px rgba(11,31,58,0.07);">
      <p style="font-size:0.7rem;letter-spacing:0.1em;text-transform:uppercase;
                color:{SLATE};margin:0 0 0.5rem 0;">Índice de Demanda Proyectado</p>
      <p style="font-family:'Cormorant Garamond',serif;font-size:3.5rem;
                font-weight:700;color:{color_hex};margin:0;line-height:1;">
        {pred_sim:.1f}
      </p>
      <p style="font-size:0.85rem;color:{SLATE};margin:0.2rem 0 0.8rem 0;">de 100</p>
      {semaforo_html(pred_sim)}
    </div>
    """, unsafe_allow_html=True)

with r2:
    # Gauge-style bar chart
    fig_g, ax_g = plt.subplots(figsize=(5.5, 2.8))
    categorias = ['Baja\n(0–40)', 'Media\n(40–60)', 'Alta\n(60–100)']
    rangos     = [40, 20, 40]
    colores_g  = ['#FEE2E2', '#FEF3C7', '#D1FAE5']
    bordes_g   = [RED, ORANGE, GREEN]

    acum = 0
    for rng, col, brd in zip(rangos, colores_g, bordes_g):
        ax_g.barh(0, rng, left=acum, height=0.5, color=col,
                  edgecolor=brd, linewidth=1.2, zorder=2)
        acum += rng

    # Needle
    ax_g.axvline(pred_sim, color=color_hex, linewidth=3, zorder=5, ymin=0.15, ymax=0.85)
    ax_g.scatter([pred_sim], [0], color=color_hex, s=180, zorder=6)

    ax_g.set_xlim(0, 100)
    ax_g.set_ylim(-0.5, 0.5)
    ax_g.set_yticks([])
    ax_g.set_xticks([0, 20, 40, 60, 80, 100])
    ax_g.set_facecolor('#FAFAF8')
    ax_g.figure.set_facecolor('#FAFAF8')
    ax_g.spines['top'].set_visible(False)
    ax_g.spines['right'].set_visible(False)
    ax_g.spines['left'].set_visible(False)
    ax_g.spines['bottom'].set_color('#E8E3DB')
    ax_g.tick_params(colors=SLATE, labelsize=8)
    ax_g.set_title(f'Posición en el espectro histórico', fontsize=9.5, color=NAVY,
                   fontweight='600', pad=8)
    for cat, pos in zip(categorias, [20, 50, 80]):
        ax_g.text(pos, 0.35, cat, ha='center', va='bottom', fontsize=7.5,
                  color=SLATE, fontweight='500')
    plt.tight_layout()
    st.pyplot(fig_g, use_container_width=True)
    plt.close()

with r3:
    st.markdown(f"""
    <div class="tooltip-card" style="height:100%;">
      <p class="label">¿Cómo leer este índice?</p>
      <p>El índice va de <strong>0 a 100</strong>, donde 100 es el pico histórico de demanda que Los Cabos ha registrado.</p>
      <p>Un valor de <strong>{pred_sim:.0f}</strong> significa que el escenario simulado se ubica en el <strong>{pred_sim:.0f}° percentil</strong> del historial de demanda.</p>
      <p style="margin:0;">Para un proyecto de <strong>+$1M USD</strong>, recomendamos lanzar en zona verde (≥60).</p>
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 2 — FORECAST
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">📈 Proyección de Demanda — Próximos 12 Meses</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">El modelo proyecta la evolución de la demanda mes a mes, manteniendo las condiciones macroeconómicas actuales.</p>', unsafe_allow_html=True)

fig_f, ax_f = plt.subplots(figsize=(13, 4.5))

# Historical (last 24 months)
hist = df['demand_index'].tail(24)
ax_f.plot(hist.index, hist.values, color=NAVY, linewidth=2.2,
          label='Demanda Histórica', zorder=3)
ax_f.fill_between(hist.index, hist.values, alpha=0.07, color=NAVY)

# Forecast
ax_f.plot(forecast_df.index, forecast_df['demand_index_pred'],
          color=GOLD, linewidth=2.5, linestyle='--',
          label='Proyección 12 meses', zorder=4)
ax_f.fill_between(forecast_df.index, forecast_df['demand_index_pred'],
                  alpha=0.18, color=GOLD)

# Zones
ax_f.axhspan(60, 105, alpha=0.06, color=GREEN)
ax_f.axhspan(40, 60,  alpha=0.06, color=ORANGE)
ax_f.axhspan(0,  40,  alpha=0.04, color=RED)
ax_f.axhline(60, color=GREEN,  linestyle=':', alpha=0.5, linewidth=1.2)
ax_f.axhline(40, color=ORANGE, linestyle=':', alpha=0.5, linewidth=1.2)

# Best month marker
ax_f.axvline(mejor_mes, color=GREEN, linewidth=1.8, linestyle=':', alpha=0.7, zorder=5)
ax_f.annotate(
    f'Mejor mes:\n{MESES_CORTO[mejor_mes.month]} {mejor_mes.year}',
    xy=(mejor_mes, demanda_forecast),
    xytext=(15, -30), textcoords='offset points',
    fontsize=8, color=GREEN, fontweight='600',
    arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2)
)

# Zone labels (right axis)
ax_f_twin = ax_f.twinx()
ax_f_twin.set_ylim(ax_f.get_ylim())
ax_f_twin.set_yticks([20, 50, 80])
ax_f_twin.set_yticklabels(['BAJA', 'MEDIA', 'ALTA'],
                          fontsize=7.5, color=SLATE, fontweight='600')
ax_f_twin.spines['right'].set_color('#E8E3DB')
ax_f_twin.tick_params(right=False)

# Divider between hist and forecast
divider_x = df.index[-1] + pd.DateOffset(days=15)
ax_f.axvline(divider_x, color=SLATE, linewidth=1, alpha=0.3, linestyle='-')
ax_f.text(divider_x, ax_f.get_ylim()[1] * 0.97, ' Proyección →',
          fontsize=7.5, color=SLATE, va='top')

set_plot_style(ax_f, ylabel='Índice de Demanda (0–100)')
ax_f.set_ylim(0, min(105, ax_f.get_ylim()[1]))
ax_f.legend(loc='lower left', fontsize=8.5, framealpha=0.9,
            edgecolor='#E8E3DB', facecolor='white')
plt.tight_layout()
st.pyplot(fig_f, use_container_width=True)
plt.close()

with st.expander('ℹ️ ¿Cómo se construye esta proyección?'):
    st.markdown(f"""
    El modelo toma las **condiciones macroeconómicas actuales** (S&P 500, confianza del consumidor, 
    tasas hipotecarias y tipo de cambio) y las mantiene constantes durante los 12 meses.  
    Lo que varía es **la estacionalidad** — el modelo aprendió que Los Cabos tiene patrones 
    de demanda muy claros según el mes del año (temporada alta: diciembre–febrero).

    El **mejor mes para lanzar** ({MESES_ES[mejor_mes.month]} {mejor_mes.year}) es el punto de 
    mayor demanda proyectada, representando una demanda de **{demanda_forecast:.1f}/100** — 
    {"en zona óptima para el lanzamiento." if demanda_forecast >= 60 else "en zona de evaluación; se recomienda monitorear evolución macro."}
    """)

# ─────────────────────────────────────────────
# SECTION 3 — ESTACIONALIDAD
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">📅 Estacionalidad — El Mejor Timing de Lanzamiento</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">¿En qué meses del año históricamente llega más el comprador objetivo?</p>', unsafe_allow_html=True)

col_s1, col_s2 = st.columns([2, 1])
with col_s1:
    demanda_mes   = df.groupby('mes')['demand_index'].mean()
    meses_nombres = [MESES_CORTO[i] for i in range(1, 13)]

    fig_s, ax_s = plt.subplots(figsize=(10, 4))
    colores_bars = [GOLD if v == demanda_mes.max() else NAVY + 'CC' for v in demanda_mes.values]
    bars = ax_s.bar(meses_nombres, demanda_mes.values, color=colores_bars,
                    width=0.65, zorder=3, edgecolor='white', linewidth=0.5)

    for bar, val in zip(bars, demanda_mes.values):
        ax_s.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                  f'{val:.0f}', ha='center', va='bottom', fontsize=8.5,
                  color=NAVY, fontweight='500')

    max_idx  = demanda_mes.values.argmax()
    ax_s.set_title(f'Demanda promedio histórica por mes — pico en {meses_nombres[max_idx]}',
                   fontsize=10, color=NAVY, fontweight='600', pad=10)
    set_plot_style(ax_s, ylabel='Índice Promedio (0–100)')
    ax_s.set_ylim(0, demanda_mes.max() * 1.18)

    gold_patch = mpatches.Patch(color=GOLD, label='Mes de mayor demanda histórica')
    navy_patch = mpatches.Patch(color=NAVY + 'CC', label='Resto del año')
    ax_s.legend(handles=[gold_patch, navy_patch], fontsize=8,
                framealpha=0.9, edgecolor='#E8E3DB')
    plt.tight_layout()
    st.pyplot(fig_s, use_container_width=True)
    plt.close()

with col_s2:
    top3 = demanda_mes.sort_values(ascending=False).head(3)
    st.markdown(f"""
    <div class="insight-box">
      <strong>¿Por qué importa esto?</strong><br><br>
      Los meses de mayor demanda son los que concentran el mayor flujo del comprador 
      objetivo — el americano y canadiense que escapa del invierno norte.<br><br>
      <strong>Top 3 meses históricos:</strong><br>
      {''.join([f"<br>• <strong>{MESES_ES[int(m)]}</strong> — índice {v:.0f}/100" for m, v in top3.items()])}<br><br>
      Lanzar en estos meses maximiza la exposición ante el comprador más activo 
      y reduce la presión de descuento en precio.
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 4 — FEATURE IMPORTANCE
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">🔍 ¿Qué Factores Mueven la Demanda en Los Cabos?</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">El modelo identificó qué variables tienen mayor peso en la predicción de demanda.</p>', unsafe_allow_html=True)

nombres_display = {
    'sp500'           : 'Bolsa de valores EE.UU. (S&P 500)',
    'confianza'       : 'Confianza del consumidor americano',
    'hipotecas'       : 'Tasa hipotecaria (30 años)',
    'tipo_cambio'     : 'Tipo de cambio Peso / Dólar',
    'mes'             : 'Mes del año (estacionalidad)',
    'trimestre'       : 'Trimestre del año',
    'turistas_lag1'   : 'Demanda del mes anterior',
    'turistas_lag3'   : 'Demanda de hace 3 meses',
    'turistas_lag12'  : 'Demanda del mismo mes, año anterior'
}
explicaciones = {
    'turistas_lag12'  : 'El patrón de demanda del año anterior se repite — Los Cabos tiene estacionalidad muy marcada.',
    'turistas_lag1'   : 'Si el mes pasado hubo alta demanda, el momentum suele continuar.',
    'sp500'           : 'Cuando la bolsa sube, el comprador americano de lujo tiene más riqueza en papel y gasta más.',
    'tipo_cambio'     : 'Un peso débil hace Los Cabos más accesible en precio para el comprador extranjero.',
    'confianza'       : 'La confianza económica anticipa el gasto discrecional en segunda vivienda.',
    'hipotecas'       : 'Tasas altas enfrían el sentimiento comprador, aunque Los Cabos es mayormente cash.',
    'mes'             : 'La temporada del año define fuertemente el apetito de viaje.',
    'turistas_lag3'   : 'Captura tendencias de mediano plazo (un trimestre atrás).',
    'trimestre'       : 'Refuerza el patrón estacional junto con el mes.'
}

importancias = rf_pipeline.named_steps['model'].feature_importances_
fi_df = pd.DataFrame({
    'variable'   : FEATURE_COLS,
    'display'    : [nombres_display[c] for c in FEATURE_COLS],
    'importance' : importancias
}).sort_values('importance', ascending=True)

fig_fi, ax_fi = plt.subplots(figsize=(10, 5.5))
colores_fi = [GOLD if v == fi_df['importance'].max() else NAVY + 'CC' for v in fi_df['importance']]
bars_fi = ax_fi.barh(fi_df['display'], fi_df['importance'],
                     color=colores_fi, height=0.55,
                     edgecolor='white', linewidth=0.5, zorder=3)

for bar, val in zip(bars_fi, fi_df['importance']):
    ax_fi.text(val + 0.003, bar.get_y() + bar.get_height()/2,
               f'{val*100:.1f}%', va='center', fontsize=8.5, color=NAVY, fontweight='500')

set_plot_style(ax_fi, xlabel='Peso relativo en el modelo')
ax_fi.set_xlim(0, fi_df['importance'].max() * 1.25)
ax_fi.set_title('Variables ordenadas por influencia en la predicción de demanda',
                fontsize=10, color=NAVY, fontweight='600', pad=10)

gold_patch2 = mpatches.Patch(color=GOLD, label='Variable más influyente')
ax_fi.legend(handles=[gold_patch2], fontsize=8, framealpha=0.9, edgecolor='#E8E3DB')
plt.tight_layout()
st.pyplot(fig_fi, use_container_width=True)
plt.close()

top_var = fi_df.iloc[-1]['variable']
st.markdown(f"""
<div class="insight-box">
  <strong>Conclusión para la toma de decisión:</strong> El factor más determinante 
  de la demanda futura es <strong>{nombres_display[top_var]}</strong>. 
  Esto confirma que Los Cabos tiene una <strong>estacionalidad muy predecible</strong>, 
  lo que facilita la planeación del timing de lanzamiento. El siguiente factor en importancia 
  son las condiciones macro de EE.UU. — lo que refuerza que el proyecto esté directamente 
  vinculado al ciclo económico del comprador norteamericano.
</div>
""", unsafe_allow_html=True)

with st.expander('📖 ¿Qué significa cada factor? (explicación de negocio)'):
    cols_exp = st.columns(2)
    for i, (var, display) in enumerate(zip(fi_df['variable'].tolist()[::-1], fi_df['display'].tolist()[::-1])):
        with cols_exp[i % 2]:
            st.markdown(f"""
            <div class="tooltip-card" style="margin-bottom:0.8rem;">
              <p class="label">{display}</p>
              <p style="margin:0;">{explicaciones.get(var, '')}</p>
            </div>
            """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 5 — METHODOLOGY (Non-technical)
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">📚 Metodología y Fuentes de Datos</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">Cómo se construyó este modelo de inteligencia de demanda.</p>', unsafe_allow_html=True)

m1, m2, m3 = st.columns(3)
with m1:
    st.markdown(f"""
    <div class="tooltip-card">
      <p class="label">🏦 Fuente 1 — FRED</p>
      <p><strong>¿Qué es?</strong> La base de datos económica abierta de la Reserva Federal de Estados Unidos. 
      Contiene miles de series históricas gratuitas.</p>
      <p><strong>¿Qué tomamos de ahí?</strong></p>
      <ul style="margin:0;padding-left:1.2rem;color:{SLATE};">
        <li>Nivel del S&P 500 (efecto riqueza)</li>
        <li>Confianza del consumidor americano</li>
        <li>Tasas hipotecarias a 30 años</li>
        <li>Tipo de cambio USD/MXN</li>
      </ul>
      <p style="margin-top:0.8rem;margin-bottom:0;">
        <span class="source-pill">FRED API</span>
        <span class="source-pill">Fed Reserve EE.UU.</span>
      </p>
    </div>
    """, unsafe_allow_html=True)

with m2:
    st.markdown(f"""
    <div class="tooltip-card">
      <p class="label">✈️ Fuente 2 — SECTUR</p>
      <p><strong>¿Qué es?</strong> La Secretaría de Turismo de México publica mensualmente 
      el volumen real de turistas por aeropuerto y país de residencia.</p>
      <p><strong>¿Qué tomamos de ahí?</strong> Las llegadas reales de turistas 
      provenientes de <strong>EE.UU. y Canadá</strong> al aeropuerto de Los Cabos — 
      exactamente el perfil del comprador objetivo de Fortem Capital.</p>
      <p><strong>¿Por qué no Google Trends?</strong> Los términos de búsqueda de bienes 
      raíces de lujo tienen volúmenes muy bajos en términos específicos para generar 
      una señal confiable mes a mes.</p>
      <p style="margin-bottom:0;">
        <span class="source-pill">SECTUR</span>
        <span class="source-pill">Datos Reales</span>
      </p>
    </div>
    """, unsafe_allow_html=True)

with m3:
    st.markdown(f"""
    <div class="tooltip-card">
      <p class="label">🤖 Modelo — Random Forest</p>
      <p><strong>¿Por qué este modelo?</strong> Piénsalo como un <em>comité de 200 expertos</em> 
      que cada uno aprende patrones distintos de los datos y después votan juntos para dar 
      una predicción final.</p>
      <p>Es ideal para este análisis porque:</p>
      <ul style="margin:0;padding-left:1.2rem;color:{SLATE};">
        <li>Funciona bien con datasets medianos (84 meses)</li>
        <li>Captura relaciones no lineales entre variables</li>
        <li>Es robusto ante datos atípicos (excluimos COVID)</li>
        <li>No requiere supuestos estadísticos rígidos</li>
      </ul>
      <p style="margin-top:0.8rem;margin-bottom:0;">
        <strong>Precisión:</strong> El modelo explica el <strong>51%</strong> de la variación 
        en demanda usando solo datos públicos — resultado sólido para un mercado de lujo 
        con datos limitados.
      </p>
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 6 — DECISION SUMMARY
# ─────────────────────────────────────────────
st.markdown('<hr class="gold-divider">', unsafe_allow_html=True)
st.markdown('<p class="section-title">🏁 Resumen para la Toma de Decisión</p>', unsafe_allow_html=True)
st.markdown('<p class="section-subtitle">Síntesis ejecutiva basada en el análisis cuantitativo.</p>', unsafe_allow_html=True)

d1, d2 = st.columns([1.6, 1])
with d1:
    semaforo_texto = (
        f"Las condiciones proyectadas para los próximos 12 meses son **favorables** para el lanzamiento. "
        f"El pico de demanda proyectado ({demanda_forecast:.1f}/100) se ubica en zona verde, "
        f"con el mejor momento de lanzamiento en **{MESES_ES[mejor_mes.month]} {mejor_mes.year}**."
    ) if demanda_forecast >= 60 else (
        f"Las condiciones proyectadas son **moderadas**. Se recomienda monitorear la evolución "
        f"del S&P 500 y la confianza del consumidor americano antes de comprometer el lanzamiento completo."
    ) if demanda_forecast >= 40 else (
        f"Las condiciones actuales sugieren **esperar**. El modelo detecta que el comprador objetivo "
        f"americano está en un ciclo de menor actividad. Se recomienda revisar en 3–6 meses."
    )

    decision_color = GREEN if demanda_forecast >= 60 else ORANGE if demanda_forecast >= 40 else RED

    st.markdown(f"""
    <div style="background:white;border-radius:12px;border:1px solid #E8E3DB;
                border-left:4px solid {decision_color};padding:1.5rem 1.8rem;
                box-shadow:0 2px 12px rgba(11,31,58,0.07);">
      <p style="font-size:0.7rem;letter-spacing:0.1em;text-transform:uppercase;
                color:{SLATE};margin:0 0 0.6rem 0;">Recomendación del Modelo</p>
      {semaforo_html(demanda_forecast)}
      <p style="margin:1rem 0 0 0;font-size:0.9rem;color:{NAVY};line-height:1.7;">{semaforo_texto}</p>

      <div style="margin-top:1.2rem;padding-top:1rem;border-top:1px solid #E8E3DB;">
        <p style="font-size:0.75rem;color:{SLATE};margin:0;">
          <strong>Supuestos del modelo:</strong> condiciones macroeconómicas actuales constantes · 
          Periodo de análisis: 2015–2024 excl. COVID · 
          Datos: FRED + SECTUR · R² = 0.51
        </p>
      </div>
    </div>
    """, unsafe_allow_html=True)

with d2:
    st.markdown(f"""
    <div style="background:{NAVY};border-radius:12px;padding:1.5rem 1.4rem;color:white;height:100%;">
      <p style="font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;
                color:{GOLD};margin:0 0 1rem 0;">Variables de Monitoreo</p>
      <p style="font-size:0.83rem;opacity:0.85;line-height:1.7;margin:0;">
        Para actualizar esta recomendación, monitorear mensualmente:
      </p>
      <ul style="font-size:0.83rem;opacity:0.85;line-height:2;margin:0.5rem 0 0 0;padding-left:1.2rem;">
        <li>S&P 500 ({'actual: {:,.0f}'.format(df['sp500'].iloc[-1])})</li>
        <li>Confianza U. of Michigan ({df['confianza'].iloc[-1]:.1f})</li>
        <li>Tasa hipotecaria 30y ({df['hipotecas'].iloc[-1]:.2f}%)</li>
        <li>USD/MXN ({df['tipo_cambio'].iloc[-1]:.2f})</li>
      </ul>
      <p style="font-size:0.78rem;opacity:0.6;margin:1.2rem 0 0 0;line-height:1.5;">
        Fuente: FRED · Actualización mensual recomendada
      </p>
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────
st.markdown(f"""
<div class="footer">
  <span class="name">Ana Paola Becerril Gutiérrez</span>
  <span class="divider">·</span>
  Análisis de Inversión · Fortem Capital
  <span class="divider">·</span>
  Los Cabos Luxury Demand Intelligence
  <br>
  <span style="font-size:0.72rem;opacity:0.55;">
    Modelo: Random Forest Regressor · Fuentes: FRED (Federal Reserve) + SECTUR México · 
    Datos: 2015–2024 (excl. COVID) · R² = 0.51
  </span>
</div>
""", unsafe_allow_html=True)


In [ ]:
#!pip install streamlit pyngrok --quiet

from pyngrok import ngrok
import subprocess, time

ngrok.kill()

subprocess.Popen(['streamlit', 'run', 'fortem_app.py',
                  '--server.port', '8501',
                  '--server.headless', 'true'])
time.sleep(4)

ngrok.set_auth_token('357d9y5SkSGvWRaY2sa0VKYVMGK_4GSaEus6iW8gEsnmUKp6w')
public_url = ngrok.connect(8501)
print('='*60)
print('🚀 DASHBOARD FORTEM CAPITAL PUBLICADO')
print(f'🌐 URL pública: {public_url}')
print('='*60)


---
# Diagnostico completo del modelo


In [ ]:
print('='*55)
print('🔬 DIAGNÓSTICO COMPLETO DEL MODELO')
print('='*55)

# 1. Métricas básicas
y_pred = rf_pipeline.predict(X_test)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = np.mean(np.abs(y_test - y_pred))

print(f'\n📊 MÉTRICAS:')
print(f'   R²   = {r2:.4f}')
print(f'   RMSE = {rmse:.2f} puntos')
print(f'   MAE  = {mae:.2f} puntos')

# 2. ¿Sobreajuste? Comparar train vs test
y_pred_train = rf_pipeline.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)
print(f'\n🎯 DETECCIÓN DE SOBREAJUSTE:')
print(f'   R² en Training: {r2_train:.4f}')
print(f'   R² en Test:     {r2:.4f}')
print(f'   Diferencia:     {r2_train - r2:.4f}', end=' ')
if r2_train - r2 > 0.3:
    print('⚠️  SOBREAJUSTE — modelo memoriza en vez de aprender')
else:
    print('✅ Sin sobreajuste significativo')

# 3. ¿El modelo es mejor que adivinar el promedio?
y_promedio = np.full_like(y_test, y_train.mean())
r2_baseline = r2_score(y_test, y_promedio)
print(f'\n📏 COMPARATIVA VS BASELINE (predecir siempre el promedio):')
print(f'   R² baseline (promedio): {r2_baseline:.4f}')
print(f'   R² nuestro modelo:      {r2:.4f}', end=' ')
if r2 > r2_baseline:
    print(f'✅ Nuestro modelo es {r2 - r2_baseline:.4f} puntos mejor que el baseline')
else:
    print('❌ El modelo no mejora al baseline')

# 4. Distribución de errores
errores = y_test.values - y_pred
print(f'\n📉 DISTRIBUCIÓN DE ERRORES:')
print(f'   Error máximo positivo: +{errores.max():.1f} puntos (subestimó demanda)')
print(f'   Error máximo negativo: {errores.min():.1f} puntos (sobreestimó demanda)')
print(f'   El 80% de predicciones tiene error menor a: {np.percentile(np.abs(errores), 80):.1f} puntos')


---
# 🎤 Guía para la entrevista

**En una oración:**
> *'Construí un Random Forest Regressor que predice la demanda de compradores de lujo en Los Cabos usando indicadores macro de EE.UU. (FRED) y llegadas reales de turistas (SECTUR), logrando un R² de 0.51 con 84 observaciones mensuales.'*

**Si preguntan por el proceso:**
> *'El modelo inicial con Google Trends dio R² negativo. Diagnostiqué que el término era demasiado nicho y que las variables macro no tenían correlación con esa señal. Cambié la Y a llegadas reales de SECTUR, excluí el periodo COVID como outlier estructural, y agregué lag features para darle memoria temporal al modelo — pasando de R² = -1.86 a R² = 0.51.'*

**Si preguntan por qué Random Forest:**
> *'Con solo 84 observaciones mensuales, Random Forest es más robusto que una red neuronal. Como siguiente paso, con datos diarios o un horizonte mayor, evaluaría LSTM para capturar dependencias temporales de largo plazo.'*

**Si preguntan por el R² de 0.51:**
> *'Para datos macroeconómicos mensuales con 84 observaciones y sin datos propietarios de ventas, un R² de 0.51 es un resultado honesto y significativo. El modelo captura el 51% de la variación en demanda usando solo variables públicas.'*
